# Backpropagation & Training with Stochastic Gradient Descent

**Problem:** We have many weights and biases tangled together across layers. How do we figure out how to adjust *each one* to reduce errors?

**Answer:** Use the **chain rule** from calculus to untangle the derivatives — this process is called **backpropagation**.

---

## The Big Picture

```
Forward:   X → Z1 → A1 → Z2 → A2 → Cost (error)
Backward:  Cost → A2 → Z2 → A1 → Z1 → adjust W and B
```

We compute the error at the output, then "propagate" it *backward* through the layers to figure out how much each weight/bias contributed to the error.

---

## Cost Function

The cost (error) for a single prediction is the **squared difference**:

$$ C = (A_2 - Y)^2 $$

Where:
- $A_2$ = our prediction (probability from the output layer)
- $Y$ = the actual answer (0 or 1)

## Chain Rule — Breaking Down the Derivatives

To find how a weight in the output layer ($W_2$) affects the cost:

$$ \frac{dC}{dW_2} = \frac{dZ_2}{dW_2} \cdot \frac{dA_2}{dZ_2} \cdot \frac{dC}{dA_2} $$

**In plain English:** A change in $W_2$ changes $Z_2$, which changes $A_2$, which changes the cost $C$. We multiply these effects together.

### Each Piece:

| Derivative | Value | Meaning |
| --- | --- | --- |
| $\frac{dC}{dA_2}$ | $2(A_2 - Y)$ | How cost changes with prediction |
| $\frac{dA_2}{dZ_2}$ | $\frac{e^{-Z_2}}{(1+e^{-Z_2})^2}$ | Derivative of sigmoid |
| $\frac{dZ_2}{dW_2}$ | $A_1$ | Slope = input to that layer |

In [1]:
from sympy import *

# --- Verify each derivative with SymPy ---

# 1. dC/dA2
A2, Y = symbols('A2 Y')
C = (A2 - Y)**2
dC_dA2 = diff(C, A2)
print("dC/dA2 =", dC_dA2)   # 2*A2 - 2*Y

# 2. dA2/dZ2 (derivative of sigmoid)
Z2 = symbols('Z2')
sigmoid = 1 / (1 + exp(-Z2))
dA2_dZ2 = diff(sigmoid, Z2)
print("dA2/dZ2 =", dA2_dZ2)  # exp(-Z2)/(1 + exp(-Z2))**2

# 3. dZ2/dW2
A1, W2, B2 = symbols('A1 W2 B2')
_Z2 = A1 * W2 + B2
dZ2_dW2 = diff(_Z2, W2)
print("dZ2/dW2 =", dZ2_dW2)  # A1

dC/dA2 = 2*A2 - 2*Y
dA2/dZ2 = exp(-Z2)/(1 + exp(-Z2))**2
dZ2/dW2 = A1


## All Four Derivatives We Need

We need gradients for **4 things**: $W_1, B_1, W_2, B_2$

**Output layer** (shorter chain):

$$ \frac{dC}{dW_2} = A_1 \cdot \frac{e^{-Z_2}}{(1+e^{-Z_2})^2} \cdot 2(A_2 - Y) $$

$$ \frac{dC}{dB_2} = 1 \cdot \frac{e^{-Z_2}}{(1+e^{-Z_2})^2} \cdot 2(A_2 - Y) $$

**Hidden layer** (longer chain — goes through output layer first):

$$ \frac{dC}{dW_1} = 2(A_2 - Y) \cdot \frac{e^{-Z_2}}{(1+e^{-Z_2})^2} \cdot W_2 \cdot [Z_1 > 0] \cdot X $$

$$ \frac{dC}{dB_1} = 2(A_2 - Y) \cdot \frac{e^{-Z_2}}{(1+e^{-Z_2})^2} \cdot W_2 \cdot [Z_1 > 0] \cdot 1 $$

The $[Z_1 > 0]$ part is the derivative of ReLU — it's 1 for positive values and 0 for negative values.

In [2]:
# All partial derivatives verified with SymPy
W1, W2, B1, B2, A1, A2, Z1, Z2, X, Y = \
    symbols('W1 W2 B1 B2 A1 A2 Z1 Z2 X Y')

# Cost function
C = (A2 - Y)**2
dC_dA2 = diff(C, A2)
print("dC/dA2 =", dC_dA2)

# Sigmoid derivative
logistic = lambda x: 1 / (1 + exp(-x))
_A2 = logistic(Z2)
dA2_dZ2 = diff(_A2, Z2)
print("dA2/dZ2 =", dA2_dZ2)

# Output layer derivatives
_Z2 = A1 * W2 + B2
print("dZ2/dA1 =", diff(_Z2, A1))  # W2
print("dZ2/dW2 =", diff(_Z2, W2))  # A1
print("dZ2/dB2 =", diff(_Z2, B2))  # 1

# ReLU derivative (manual — not smooth, so can't use diff)
d_relu = lambda x: x > 0  # 1 if positive, 0 if negative
print("dA1/dZ1 = Z1 > 0  (1 if positive, 0 if negative)")

# Hidden layer derivatives
_Z1 = X * W1 + B1
print("dZ1/dW1 =", diff(_Z1, W1))  # X
print("dZ1/dB1 =", diff(_Z1, B1))  # 1

dC/dA2 = 2*A2 - 2*Y
dA2/dZ2 = exp(-Z2)/(1 + exp(-Z2))**2
dZ2/dA1 = W2
dZ2/dW2 = A1
dZ2/dB2 = 1
dA1/dZ1 = Z1 > 0  (1 if positive, 0 if negative)
dZ1/dW1 = X
dZ1/dB1 = 1


## Full Training — Putting It All Together

Now we combine:
1. **Forward propagation** — get predictions
2. **Backpropagation** — compute gradients using chain rule
3. **Gradient descent** — nudge weights/biases in the direction that reduces error

We repeat this for **100,000 iterations**, sampling 1 random training example per iteration.

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Load data
all_data = pd.read_csv("https://tinyurl.com/y2qmhfsr")

# Learning rate — controls step size
# Too small = slow training, too big = overshoots
L = 0.05

# Prepare data
all_inputs = all_data.iloc[:, 0:3].values / 255.0
all_outputs = all_data.iloc[:, -1].values
X_train, X_test, Y_train, Y_test = train_test_split(
    all_inputs, all_outputs, test_size=1/3
)
n = X_train.shape[0]

# Initialize random weights and biases
w_hidden = np.random.rand(3, 3)
w_output = np.random.rand(1, 3)
b_hidden = np.random.rand(3, 1)
b_output = np.random.rand(1, 1)

# Activation functions
relu = lambda x: np.maximum(x, 0)
logistic = lambda x: 1 / (1 + np.exp(-x))

# Derivatives of activation functions
d_relu = lambda x: x > 0                                # 1 if positive, 0 if negative
d_logistic = lambda x: np.exp(-x) / (1 + np.exp(-x))**2  # sigmoid derivative

In [4]:
# Forward propagation
def forward_prop(X):
    Z1 = w_hidden @ X + b_hidden
    A1 = relu(Z1)
    Z2 = w_output @ A1 + b_output
    A2 = logistic(Z2)
    return Z1, A1, Z2, A2

# Backpropagation — returns slopes for all weights and biases
def backward_prop(Z1, A1, Z2, A2, X, Y):
    dC_dA2 = 2 * A2 - 2 * Y             # cost derivative
    dA2_dZ2 = d_logistic(Z2)             # sigmoid derivative
    dZ2_dA1 = w_output                   # weights connecting hidden → output
    dZ2_dW2 = A1                         # input to output layer
    dZ2_dB2 = 1                          # bias derivative is always 1
    dA1_dZ1 = d_relu(Z1)                 # ReLU derivative
    dZ1_dW1 = X                          # input to hidden layer
    dZ1_dB1 = 1

    # Chain rule — output layer gradients
    dC_dW2 = dC_dA2 @ dA2_dZ2 @ dZ2_dW2.T
    dC_dB2 = dC_dA2 @ dA2_dZ2 * dZ2_dB2

    # Chain rule — hidden layer gradients (longer chain)
    dC_dA1 = dC_dA2 @ dA2_dZ2 @ dZ2_dA1
    dC_dW1 = dC_dA1 @ dA1_dZ1 @ dZ1_dW1.T
    dC_dB1 = dC_dA1 @ dA1_dZ1 * dZ1_dB1

    return dC_dW1, dC_dB1, dC_dW2, dC_dB2

In [5]:
# --- TRAINING LOOP ---
# 100,000 iterations of stochastic gradient descent

for i in range(100_000):
    # Pick 1 random training sample
    idx = np.random.choice(n, 1, replace=False)
    X_sample = X_train[idx].transpose()
    Y_sample = Y_train[idx]

    # Forward pass — get predictions
    Z1, A1, Z2, A2 = forward_prop(X_sample)

    # Backward pass — get gradients
    dW1, dB1, dW2, dB2 = backward_prop(Z1, A1, Z2, A2, X_sample, Y_sample)

    # Update weights and biases (nudge in direction of lower cost)
    w_hidden -= L * dW1
    b_hidden -= L * dB1
    w_output -= L * dW2
    b_output -= L * dB2

print("Training complete!")

Training complete!


In [6]:
# --- EVALUATE ---
test_predictions = forward_prop(X_test.transpose())[3]  # grab A2
test_comparisons = np.equal(
    (test_predictions >= 0.5).flatten().astype(int),
    Y_test
)
accuracy = sum(test_comparisons.astype(int) / X_test.shape[0])

print(f"Test Accuracy: {accuracy:.2%}")
print("\n(Should be around 97-99% after training!)")

Test Accuracy: 98.89%

(Should be around 97-99% after training!)


## Interactive Demo — Try Your Own Colors!

After training, we can predict light/dark font for any background color.

In [7]:
def predict_font(r, g, b):
    """Predict whether to use light or dark font for a given RGB background color."""
    X = np.array([[r, g, b]]).transpose() / 255.0
    _, _, _, A2 = forward_prop(X)
    prob = A2[0][0]
    shade = "LIGHT font" if prob >= 0.5 else "DARK font"
    return f"RGB({r},{g},{b}) → {shade} (confidence: {prob:.4f})"

# Test with some colors
print(predict_font(0, 0, 0))        # Black → should be LIGHT font
print(predict_font(255, 255, 255))  # White → should be DARK font
print(predict_font(255, 140, 0))    # Dark orange
print(predict_font(255, 192, 203))  # Pink
print(predict_font(0, 0, 128))      # Navy blue

RGB(0,0,0) → DARK font (confidence: 0.0089)
RGB(255,255,255) → LIGHT font (confidence: 1.0000)
RGB(255,140,0) → LIGHT font (confidence: 1.0000)
RGB(255,192,203) → LIGHT font (confidence: 1.0000)
RGB(0,0,128) → DARK font (confidence: 0.0089)


## Key Takeaways

1. **Backpropagation** = using the chain rule to find how each weight/bias affects the final error
2. We compute gradients **backwards** from the output layer to the hidden layer
3. **Gradient descent** then nudges each weight/bias by `learning_rate × gradient`
4. After 100,000 iterations of SGD, accuracy jumps from ~55% to **97–99%**
5. The **learning rate (L)** is critical — too small = slow, too big = overshoots the solution